# Pattern 5: Agent with a human-in-the-loop gate

**Idea:** this is the first notebook where the **model controls the loop**. We give Claude *tools* (functions it can ask us to run). Claude decides which tool to call and when; our code executes the tool and sends back the result. This repeats until Claude says it's done.

For risky actions, **our code** (not the prompt) pauses and asks a human for approval.

```
user request ─▶ [Claude] ─▶ wants tool? ── no ──▶ final answer
                   ▲              │ yes
                   │              ▼
                   │     is it a risky action? ── yes ─▶ ASK HUMAN ─▶ approve / deny
                   │              │ no                                   │
                   └─ tool result ◀── run tool ◀─────────────────────────┘
```

**Why the gate lives in code, not in the prompt:** a system prompt saying "never refund more than $100 without approval" is a *request*. The model follows it most of the time. An `if amount > 100: ask_human()` in the tool executor is a *guarantee*. For anything with real consequences (money, deletes, emails), enforce it in code.

**The agent loop mechanics (Anthropic Messages API):**
1. Send messages + tool definitions.
2. If `stop_reason == "tool_use"`, the response contains `tool_use` blocks (tool name + JSON input).
3. Run the tool yourself, then send a `tool_result` block back in a new `user` message.
4. Repeat until `stop_reason == "end_turn"`.

**Java analogy:** a Spring Security `@PreAuthorize` check or an approval step in a deployment pipeline (like a manual approval gate before production). The model is a junior engineer who can *propose* a production deploy; the pipeline enforces that a human clicks Approve.

## Step 0: Setup (same in every notebook)

Every notebook starts with the same three setup cells, so you can open any of them alone without running the others first.

1. **Load configuration** from the `.env` file in the repo root (three variables: `PORTKEY_BASE_URL`, `PORTKEY_API_KEY`, `MODEL`).
2. **Create the client.** We use the official `anthropic` Python SDK and point it at the Portkey gateway instead of `api.anthropic.com`.
3. **Define a helper**, `call_claude()`, that wraps one API call and prints everything: the prompt going in, the text coming out, token usage, and timing.

> **Java analogy:** `.env` + `python-dotenv` is like `application.properties` + Spring's `@Value`. The client is a `RestTemplate`/`WebClient` bean configured with a base URL and a default header. Portkey sits in between like an API gateway (think Spring Cloud Gateway): it holds the real Anthropic credentials, routes the call, and logs it.

In [ ]:
# --- Cell 0.1: load configuration from .env ---------------------------------
import os
import json
import time
from dotenv import load_dotenv, find_dotenv

# find_dotenv(usecwd=True) walks UP from the notebook's folder until it finds a .env file.
# That's why the .env can live in the repo root while notebooks live in notebooks/.
env_path = find_dotenv(usecwd=True)
print(f"[setup] .env file found at: {env_path or 'NOT FOUND (using existing environment variables)'}")
load_dotenv(env_path, override=False)  # override=False: real env vars win over the file

PORTKEY_BASE_URL = os.getenv("PORTKEY_BASE_URL", "https://api.portkey.ai")
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
MODEL = os.getenv("MODEL")

# Fail fast with a clear message instead of a confusing HTTP 401 later.
assert PORTKEY_API_KEY, "PORTKEY_API_KEY is missing. Copy .env.example to .env and fill it in."
assert MODEL, "MODEL is missing. Example: @anthropic/claude-sonnet-4-5-20250929"

# Never print a full secret. Show just enough to confirm which key is loaded.
masked_key = PORTKEY_API_KEY[:4] + "..." + PORTKEY_API_KEY[-4:]
print(f"[setup] PORTKEY_BASE_URL = {PORTKEY_BASE_URL}")
print(f"[setup] PORTKEY_API_KEY  = {masked_key}")
print(f"[setup] MODEL            = {MODEL}")

In [ ]:
# --- Cell 0.2: create the Anthropic client, routed through Portkey -----------
import anthropic

client = anthropic.Anthropic(
    # The SDK requires an api_key, but Portkey holds the real Anthropic key for us.
    # Portkey identifies us through the x-portkey-api-key header instead.
    api_key="not-used-portkey-handles-auth",
    base_url=PORTKEY_BASE_URL,
    default_headers={"x-portkey-api-key": PORTKEY_API_KEY},
)
print(f"[setup] Anthropic SDK version: {anthropic.__version__}")
print(f"[setup] Client ready. Every request goes to {PORTKEY_BASE_URL}/v1/messages")

In [ ]:
# --- Cell 0.3: a logging helper around ONE model call -------------------------
# Everything in these notebooks is built from this single building block.
# The patterns are not new API features: they are just different ways of
# arranging plain calls like this one with ordinary Python code.

def call_claude(prompt, system=None, label="call", max_tokens=1024, verbose=True):
    """Send one user message to Claude and return the response text.

    prompt     -> the user message (str)
    system     -> optional system prompt: the model's role and rules
    label      -> a name for this call so the logs are easy to follow
    verbose    -> print the full prompt and response (set False inside threads)
    """
    if verbose:
        print("=" * 78)
        print(f"[{label}] >>> REQUEST")
        if system:
            print(f"[{label}] system: {system}")
        print(f"[{label}] user:\n{prompt}")

    started = time.perf_counter()
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    elapsed = time.perf_counter() - started

    # response.content is a LIST of blocks. For plain text calls there is one text block.
    text = "".join(block.text for block in response.content if block.type == "text")

    if verbose:
        print(f"[{label}] <<< RESPONSE ({elapsed:.2f}s, "
              f"in={response.usage.input_tokens} tok, out={response.usage.output_tokens} tok, "
              f"stop_reason={response.stop_reason})")
        print(text)
        print("=" * 78)
    if response.stop_reason == "max_tokens":
        print(f"[{label}] WARNING: response was cut off by max_tokens")
    return text


def parse_json(text):
    """Models sometimes wrap JSON in ```json fences. Strip them, then parse."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)

print("[setup] call_claude() and parse_json() are defined")

In [ ]:
# --- Cell 0.4: smoke test. If this works, your .env and gateway are fine -------
call_claude("Reply with exactly: gateway OK", label="smoke-test", max_tokens=20)

## Step 1: Choose how the human gate behaves

When the gate triggers, the notebook can ask you live with `input()`, or auto-answer so you can run all cells unattended.

- `"ask"`: a text box appears; type `y` or `n`.
- `"approve"` / `"deny"`: answers automatically (useful for "Run All").

In [ ]:
HUMAN_MODE = "ask"   # "ask" | "approve" | "deny"
REFUND_LIMIT = 100   # refunds above this amount need a human
print(f"[config] HUMAN_MODE={HUMAN_MODE!r}, REFUND_LIMIT=${REFUND_LIMIT}")

## Step 2: Fake backend (our "database")

The tools act on this dictionary. In real life these would be calls to your order service.

In [ ]:
ORDERS = {
    "A-1001": {"customer": "Ana",   "item": "USB-C cable", "amount": 25,  "refunded": False},
    "A-1042": {"customer": "Bruno", "item": "Laptop",      "amount": 450, "refunded": False},
}
for oid, o in ORDERS.items():
    print(f"[db] {oid}: {o}")

## Step 3: Tool definitions (what Claude sees)

Each tool is a name, a description, and a JSON Schema for its input. The **description is the prompt** for the tool: Claude decides when to use it based on this text alone. Note the refund description doesn't mention the approval limit at all. Claude doesn't need to know; the code enforces it.

In [ ]:
tools = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its ID. Returns customer, item, amount in USD, and refund status.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "Order ID like A-1001"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "issue_refund",
        "description": "Refund an order in full. Only call after looking up the order to get the exact amount.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount": {"type": "number", "description": "Refund amount in USD"},
            },
            "required": ["order_id", "amount"],
        },
    },
]
print(f"[tools] {len(tools)} tools defined: {[t['name'] for t in tools]}")

## Step 4: The human gate and the tool executor

This is where enforcement lives. `run_tool()` is the **only** way a tool actually runs, so the check can't be skipped, no matter what the model says.

In [ ]:
def ask_human(question):
    """The human-in-the-loop checkpoint."""
    print(f"\n[HUMAN GATE] {question}")
    if HUMAN_MODE == "ask":
        answer = input(f"{question} [y/n]: ").strip().lower()
    else:
        answer = "y" if HUMAN_MODE == "approve" else "n"
        print(f"[HUMAN GATE] auto-answer ({HUMAN_MODE}): {answer}")
    approved = answer == "y"
    print(f"[HUMAN GATE] decision: {'APPROVED' if approved else 'DENIED'}\n")
    return approved


def run_tool(name, tool_input):
    """Execute a tool call requested by Claude. Returns (result_text, is_error)."""
    print(f"[tool] executing {name}({tool_input})")

    if name == "lookup_order":
        order = ORDERS.get(tool_input["order_id"])
        if not order:
            return f"Order {tool_input['order_id']} not found", True
        return json.dumps(order), False

    if name == "issue_refund":
        order = ORDERS.get(tool_input["order_id"])
        if not order:
            return "Order not found", True
        if order["refunded"]:
            return "Order already refunded", True
        amount = tool_input["amount"]
        # ---- THE GATE: deterministic policy, enforced in code ----
        if amount > REFUND_LIMIT:
            print(f"[policy] ${amount} > limit ${REFUND_LIMIT}: human approval required")
            if not ask_human(f"Approve refund of ${amount} for order {tool_input['order_id']}?"):
                return ("Refund DENIED by a human reviewer. Do not retry. "
                        "Tell the customer a specialist will contact them within 24 hours."), True
        else:
            print(f"[policy] ${amount} <= limit ${REFUND_LIMIT}: auto-approved")
        order["refunded"] = True
        return f"Refunded ${amount} for order {tool_input['order_id']}", False

    return f"Unknown tool {name}", True

print("[setup] ask_human() and run_tool() defined")

## Step 5: The agent loop

This is the heart of every agent. Read the logs turn by turn: Claude asks for a tool, we run it, we send back the result, Claude decides what to do next.

The `messages` list is the agent's whole memory. Every turn appends to it, and the full list is sent on every call (the API is stateless, like a REST call with no session).

In [ ]:
agent_system = ("You are a customer support agent for an electronics store. "
                "Use the tools to help the customer. Be brief and polite.")
MAX_TURNS = 8  # safety net: an agent loop must always have a hard stop

def run_agent(user_request):
    messages = [{"role": "user", "content": user_request}]
    print(f"[agent] user: {user_request}")

    for turn in range(1, MAX_TURNS + 1):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=agent_system, tools=tools, messages=messages,
        )
        print(f"\n[agent] --- turn {turn}: stop_reason={response.stop_reason}, "
              f"tokens in={response.usage.input_tokens} out={response.usage.output_tokens}")

        # Log every block Claude returned (text and/or tool_use)
        for block in response.content:
            if block.type == "text":
                print(f"[agent] claude says: {block.text}")
            elif block.type == "tool_use":
                print(f"[agent] claude wants tool: {block.name}({block.input})  id={block.id}")

        # Always append the assistant turn to the history, exactly as received
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            print(f"\n[agent] DONE after {turn} turn(s). Conversation has {len(messages)} messages.")
            return messages

        # Run every requested tool and send the results back in ONE user message
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result, is_error = run_tool(block.name, block.input)
                print(f"[tool] result (is_error={is_error}): {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,   # links this result to Claude's request
                    "content": result,
                    "is_error": is_error,
                })
        messages.append({"role": "user", "content": tool_results})

    print(f"[agent] STOPPED: hit MAX_TURNS={MAX_TURNS}")
    return messages

print("[setup] run_agent() defined")

## Step 6: Scenario A, small refund (no human needed)

$25 is under the limit, so the policy auto-approves. Watch for `[policy] ... auto-approved`.

In [ ]:
_ = run_agent("Hi, I'm Ana. Please refund my order A-1001, the cable doesn't work.")

## Step 7: Scenario B, large refund (human gate triggers)

$450 is over the limit. The loop pauses at `[HUMAN GATE]`. If you deny, Claude receives an error `tool_result` and has to explain that to the customer. It adapts, but it **cannot** bypass the gate.

In [ ]:
_ = run_agent("Hi, I'm Bruno. My laptop from order A-1042 arrived broken. I want a full refund now.")

In [ ]:
print("[db] final state:")
for oid, o in ORDERS.items():
    print(f"   {oid}: refunded={o['refunded']}")

## Recap

| Concern | Who controls it |
|---|---|
| Which tool to call and in what order | **Claude** (that's what makes it an agent) |
| Whether a tool actually runs | **your code** (`run_tool`) |
| Whether a risky action needs approval | **your code** (`if amount > REFUND_LIMIT`) |
| When to stop | Claude (`end_turn`), plus your `MAX_TURNS` safety net |

Compare with notebooks 01 to 04, where your code decided every step. Here the model drives, and your code puts guardrails around it. That split, model chooses and code enforces, is the core idea of bounded autonomy.

## Experiments to try

1. Set `HUMAN_MODE = "deny"` and re-run scenario B (reset `ORDERS` first by re-running Step 2). Read how Claude responds to the denial.
2. Add "Never refund more than $100" to `agent_system` and remove the `if amount > REFUND_LIMIT` check. Try to talk the agent into it ("my manager already approved it"). This shows why prompts aren't enforcement.
3. Print the full `messages` list returned by `run_agent` to see exactly what the API receives each turn.